In [1]:
!git clone https://github.com/FerrariKazu/Adversarial-Cognitive-Model.git


Cloning into 'Adversarial-Cognitive-Model'...
remote: Enumerating objects: 2594, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 2594 (delta 119), reused 121 (delta 62), pack-reused 2410 (from 2)
Receiving objects: 100% (2594/2594), 175.51 MiB | 28.16 MiB/s, done.
Resolving deltas: 100% (1507/1507), done.
Updating files: 100% (974/974), done.


In [2]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GIT_TOKEN")
secret_value_1 = user_secrets.get_secret("GIT_USER")
secret_value_2 = user_secrets.get_secret("github_token")
secret_value_3 = user_secrets.get_secret("github_username")
secret_value_4 = user_secrets.get_secret("HF_TOKEN")
import os
user_secrets = UserSecretsClient()
# Inject HF Token into the environment so training scripts can access it for async uploads
try:
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print(">>> HF_TOKEN successfully injected.")
except Exception as e:
    print(f">>> Warning: Could not inject HF_TOKEN: {e}")
!cd Adversarial-Cognitive-Model && git pull


>>> HF_TOKEN successfully injected.
Already up to date.


In [3]:
#!/usr/bin/env python3
"""
Kaggle 2xT4 Notebook Execution Pipeline for Sprint 2 (Synthetic Generation & Filtering)
=======================================================================================
This script automates Sprint 2 Phase A (SDXL Turbo generation on T4x2) and Phase B
(CLIP quality & diversity filtering + HuggingFace dataset upload).

It is formatted with '# %%' markers for direct execution in Kaggle Notebook cells.
"""

# %% [markdown]
# # Step 1: Environment Setup & Dependencies
# Installs packages needed for SDXL Turbo generation, CLIP filtering, and WebDataset packaging.

# %%
import os
import sys
import subprocess
import shutil
import time

# Fetch HF_TOKEN from environment or Kaggle secrets
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✓ HF_TOKEN successfully loaded and injected into environment.")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DIFFUSERS_NO_PROGRESS_BAR"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

def run_command(cmd, shell=True):
    print(f"\n[RUNNING]: {cmd}")
    result = subprocess.run(cmd, shell=shell, check=True, text=True)
    return result.returncode

print("\n>>> Installing Python Dependencies for Sprint 2...")
run_command("pip install --quiet --upgrade pip setuptools wheel")
# diffusers + transformers + accelerate for SDXL Turbo; webdataset for shard I/O
run_command("pip install --quiet diffusers transformers accelerate webdataset")
# CLIP for Phase B quality/diversity filtering (openai/CLIP — no PyPI release)
run_command("pip install --quiet git+https://github.com/openai/CLIP.git")
# opencv-python, datasets, huggingface_hub are standard; note: PIL is the import
# name — the PyPI package is Pillow (usually pre-installed on Kaggle, listed just in case)
run_command("pip install --quiet opencv-python datasets huggingface_hub Pillow")

# %% [markdown]
# # Step 2: Clone and Sync Repository
# Syncs latest code from GitHub to `/kaggle/working/Adversarial-Cognitive-Model`.

# %%
REPO_NAME = 'Adversarial-Cognitive-Model'
REPO_URL = f'https://github.com/FerrariKazu/{REPO_NAME}.git'

os.chdir('/kaggle/working')

if not os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    print('Cloning repository...')
    subprocess.run(f'git clone {REPO_URL}', shell=True, check=True)

os.chdir(f'/kaggle/working/{REPO_NAME}')
print('Syncing repository to latest commit...')
subprocess.run('git fetch origin main && git reset --hard origin/main', shell=True, check=True)

# Set PYTHONPATH
os.environ["PYTHONPATH"] = f"/kaggle/working/{REPO_NAME}:{os.environ.get('PYTHONPATH', '')}"
print(f"Working directory successfully set to: {os.getcwd()}")

# %% [markdown]
# # Step 3: Car-Only Regeneration with Improved Prompts + More Steps
# Car had 12.8% pass rate at threshold 0.25 — prompts were too elaborate for SDXL
# Turbo at 1 step. Regenerate with simpler concrete prompts and 3 inference steps.
# 
# ~4.5 hours on T4 for 20K images at 3 steps. Other classes are not regenerated
# (their raw data from the first pass is expected in ./data/synthetic_stl10_raw).

# %%
raw_output_dir = "./data/synthetic_stl10_raw"
os.makedirs(raw_output_dir, exist_ok=True)

print("--> Pre-downloading SDXL Turbo pipeline to HuggingFace cache...")
run_command("python3 -c \"from diffusers import AutoPipelineForText2Image; AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', variant='fp16', low_cpu_mem_usage=True)\"")

print("\n============================================================")
print("  Sprint 2 Phase A.5: Car-Only Regeneration (3 steps, simpler prompts)")
print("============================================================")
cmd_car = f"CUDA_VISIBLE_DEVICES=0 python3 data_generation/generate_synthetic_stl10.py --output-dir {raw_output_dir} --class-index 2 --target-per-class 20000"
print("Regenerating car (class-index 2, target 20K raw for ~2-4K filtered at threshold 0.30)...")
subprocess.run(cmd_car, shell=True, check=True)
print("✓ Car regeneration complete.")

# %% [markdown]
# # Step 4: Phase B — CLIP Quality Gate at Threshold 0.30
# Filters ALL raw data (car from new generation + other classes from first pass)
# at the recalibrated threshold 0.30 (was 0.25, which let through 100% on some classes).

# %%
print("\n============================================================")
print("  Sprint 2 Phase B: CLIP Quality & Diversity Filtering (threshold=0.30)")
print("============================================================")

filtered_output_dir = "./data/synthetic_stl10_filtered"
cmd_filter = f"python3 data_generation/filter_synthetic_clip.py --input-dir {raw_output_dir} --output-dir {filtered_output_dir} --sim-threshold 0.30"

run_command(cmd_filter)

# %% [markdown]
# # Step 5: Phase B — HuggingFace Dataset Upload
# Uploads filtered shards to HuggingFace dataset repo `FerrariKazu/stl10-synthetic`.

# %%
print("\n============================================================")
print("  Sprint 2 Phase B: Uploading Dataset to HuggingFace")
print("============================================================")

cmd_upload = f"python3 data_generation/upload_synthetic_hf.py --input-dir {filtered_output_dir} --repo-id FerrariKazu/stl10-synthetic"

run_command(cmd_upload)

print("\n🎉 Sprint 2 Kaggle Pipeline Execution Completed Successfully!")


✓ HF_TOKEN successfully loaded and injected into environment.

>>> Installing Python Dependencies for Sprint 2...

[RUNNING]: pip install --quiet --upgrade pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
torch 2.10.0+cu128 requires cuda-bindings==12.9.4; platform_system == "Linux", but you have cuda-bindings 13.2.0 which is incompatible.



[RUNNING]: pip install --quiet diffusers transformers accelerate webdataset


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav


[RUNNING]: pip install --quiet git+https://github.com/openai/CLIP.git

[RUNNING]: pip install --quiet opencv-python datasets huggingface_hub Pillow
Syncing repository to latest commit...


...

    ✓ Pipeline ready.

[-->] Generating class 'car' (0/20000)...


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


  Progress 'car':     4/20000  |  Shard 000 (4/1000)  |  Est Speed: 3530 imgs/hr
  Progress 'car':     8/20000  |  Shard 000 (8/1000)  |  Est Speed: 4106 imgs/hr
  Progress 'car':    12/20000  |  Shard 000 (12/1000)  |  Est Speed: 4340 imgs/hr
  Progress 'car':    16/20000  |  Shard 000 (16/1000)  |  Est Speed: 4465 imgs/hr
  Progress 'car':    20/20000  |  Shard 000 (20/1000)  |  Est Speed: 4541 imgs/hr
  Progress 'car':    24/20000  |  Shard 000 (24/1000)  |  Est Speed: 4586 imgs/hr
  Progress 'car':    28/20000  |  Shard 000 (28/1000)  |  Est Speed: 4614 imgs/hr
  Progress 'car':    32/20000  |  Shard 000 (32/1000)  |  Est Speed: 4632 imgs/hr
  Progress 'car':    36/20000  |  Shard 000 (36/1000)  |  Est Speed: 4643 imgs/hr
  Progress 'car':    40/20000  |  Shard 000 (40/1000)  |  Est Speed: 4648 imgs/hr
  Progress 'car':    44/20000  |  Shard 000 (44/1000)  |  Est Speed: 4649 imgs/hr
  Progress 'car':    48/20000  |  Shard 000 (48/1000)  |  Est Speed: 4650 imgs/hr
  Progress 'car': 

100%|███████████████████████████████████████| 338M/338M [00:10<00:00, 35.3MiB/s]


    ✓ CLIP model ready.

  Sprint 2 Phase B: Quality & Diversity Filtering
  Input: ./data/synthetic_stl10_raw  --> Output: ./data/synthetic_stl10_filtered
  Similarity Threshold: 0.3

[!] No raw shards found for class 'airplane'. Skipping.

[!] No raw shards found for class 'bird'. Skipping.

[-->] Filtering Class 'car'...
  Quality Gate: 0/20000 passed (Similarity >= 0.30, Pass Rate: 0.0%)

[!] No raw shards found for class 'cat'. Skipping.

[!] No raw shards found for class 'deer'. Skipping.

[!] No raw shards found for class 'dog'. Skipping.

[!] No raw shards found for class 'horse'. Skipping.

[!] No raw shards found for class 'monkey'. Skipping.

[!] No raw shards found for class 'ship'. Skipping.

[!] No raw shards found for class 'truck'. Skipping.

  Filtering Summary Report
  Class        |    Raw | Passed |  Pass % | Diversity | Flagged
  ------------------------------------------------------------
  car          |  20000 |      0 |    0.0% |    0.0000 |      OK

Report sav